In [1]:
import pandas as pd           # data mnipulation
import numpy as np            # number manipulation/crunching
import matplotlib.pyplot as plt  # plotting
# Classification report
from sklearn.metrics import  accuracy_score, classification_report 
import openpyxl
# Train Test split
from sklearn.model_selection import train_test_split
# Random forest classifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


diabetes = pd.read_csv("diabetes.csv")
diabetes['BloodPressure'] = diabetes['BloodPressure'].replace(to_replace=0,value=diabetes['BloodPressure'].median())
diabetes['BMI'] = diabetes['BMI'].replace(to_replace=0,value=diabetes['BMI'].median())
diabetes['Insulin'] = diabetes['Insulin'].replace(to_replace=0,value=diabetes['Insulin'].median())
diabetes['Glucose'] = diabetes['Glucose'].replace(to_replace=0,value=diabetes['Glucose'].median())
diabetes['SkinThickness'] = diabetes['SkinThickness'].replace(to_replace=0,value=diabetes['SkinThickness'].median())
diabetes["Outcome"].value_counts()


Y = diabetes['Outcome']
target_names=["No Diabetes", "Diabetes"]
X = diabetes[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
       'DiabetesPedigreeFunction', 'Age']]
X_featurenames = X.columns

# Split the data into train and test data:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, shuffle = True)
print(X_test.iloc[0])

# Build the model with the random forest regression algorithm:
# model = RandomForestClassifier(max_depth = 20, random_state = 0, n_estimators = 10000)
# model.fit(X_train, Y_train)
# pred = model.predict(X_test)
# accuracy = accuracy_score(Y_test, pred)
# print("Accuracy:", accuracy)
# print(classification_report(Y_test, pred, target_names=["No Diabetes", "Diabetes"]))
# #feat_importances = pd.Series(model.feature_importances_, index = X_featurenames)
# #feat_importances.nlargest(5).plot(kind = 'barh')

#XGBoost
model = xgb.XGBClassifier()
model.fit(X_train, Y_train)
pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, pred)
print("Accuracy:", accuracy)
print(classification_report(Y_test, pred, target_names=["No Diabetes", "Diabetes"]))

import lime
import lime.lime_tabular

dataframe_columns = []
test_index = "test_index"
dataframe_columns.append(test_index)
for feature in X_featurenames:
    feature_lower_interval = feature + "_lower_interval"
    feature_upper_interval = feature + "_upper_interval"
    feature_probability = feature + "_probability"
    feature_sign = feature + "_sign"
    feature_rank = feature + "_rank"
    dataframe_columns.append(feature_lower_interval)
    dataframe_columns.append(feature_upper_interval)
    dataframe_columns.append(feature_probability)
    dataframe_columns.append(feature_sign)
    dataframe_columns.append(feature_rank)
probability_no_diabetes = "probability_no_diabetes"
probability_diabetes = "probability_diabetes"
true_class = "true_class"
dataframe_columns.append(probability_no_diabetes)
dataframe_columns.append(probability_diabetes)
dataframe_columns.append(true_class)
df = pd.DataFrame(columns = dataframe_columns)
print(df)

number_iterations = 10
for i in range(X_test.shape[0]):
    for count in range(number_iterations):
        model = xgb.XGBClassifier()
        model.fit(X_train.values, Y_train.values)
        explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names = X_featurenames, class_names = ['No Diabetes', 'Diabetes'], feature_selection = "lasso_path", discretize_continuous = True, discretizer = "quartile", verbose = True, mode = 'classification')
        exp = explainer.explain_instance(X_test.iloc[i], model.predict_proba)
        #exp.show_in_notebook(show_table = True, show_all = True)
        list = exp.as_list()
        row = []
        row.append(i)
        for feature in X_featurenames:
            #print(feature)
            for j in range(len(X_featurenames)):
                if list[j][0].find(feature) != -1 :
                    #print(list[j][0])
                    probability = list[j][1]
                    rank = j + 1
                    if probability > 0 :
                        sign = "positive"
                    else :
                        sign = "negative"
                    if list[j][0].count('.') == 2:
                        lower_interval = list[j][0][0 : list[j][0].find('<')]
                        upper_interval = list[j][0][list[j][0].find('=') + 2 : len(list[j][0]) + 1]
                    elif list[j][0].count('.') == 1 and list[j][0].find('<') != -1:
                        lower_interval = 0
                        if(list[j][0].find('=') != -1):
                            upper_interval = list[j][0][list[j][0].find('=') + 2 : len(list[j][0]) + 1]
                        else:
                            upper_interval = list[j][0][list[j][0].find('<') + 2 : len(list[j][0]) + 1]
                    elif list[j][0].count('.') == 1 and list[j][0].find('>') != -1:
                        upper_interval = max(X_test[feature])
                        if(list[j][0].find('=') != -1):
                            lower_interval = list[j][0][list[j][0].find('=') + 2 : len(list[j][0]) + 1]
                        else:
                            lower_interval = list[j][0][list[j][0].find('>') + 2 : len(list[j][0]) + 1]
                    #print(lower_interval)
                    #print(upper_interval)
                    row.append(lower_interval)
                    row.append(upper_interval)
                    row.append(probability)
                    row.append(sign)
                    row.append(rank)
        probability_no_diabetes = model.predict_proba([X_test.iloc[i]])[0,0]
        probability_diabetes = model.predict_proba([X_test.iloc[i]])[0,1]
        true_class = target_names[Y_test.iloc[i]]
        row.append(probability_no_diabetes)
        row.append(probability_diabetes)
        row.append(true_class)
        #print(row)
        df.loc[i * number_iterations + count] = row
    
df.to_excel("outputDiabetesLIME.xlsx")
# wb = openpyxl.load_workbook('outputDiabetes.xlsx')
# ws = wb.active
# ws.append(row)
# wb.save('outputDiabetes.xlsx')

Pregnancies                  5.000
Glucose                     96.000
BloodPressure               74.000
SkinThickness               18.000
Insulin                     67.000
BMI                         33.600
DiabetesPedigreeFunction     0.997
Age                         43.000
Name: 265, dtype: float64
Accuracy: 0.7207792207792207
              precision    recall  f1-score   support

 No Diabetes       0.80      0.79      0.79       105
    Diabetes       0.56      0.57      0.57        49

    accuracy                           0.72       154
   macro avg       0.68      0.68      0.68       154
weighted avg       0.72      0.72      0.72       154

Empty DataFrame
Columns: [test_index, Pregnancies_lower_interval, Pregnancies_upper_interval, Pregnancies_probability, Pregnancies_sign, Pregnancies_rank, Glucose_lower_interval, Glucose_upper_interval, Glucose_probability, Glucose_sign, Glucose_rank, BloodPressure_lower_interval, BloodPressure_upper_interval, BloodPressure_probability,

KeyboardInterrupt: 